<a href="https://colab.research.google.com/github/joygoswaminiloy2023-droid/Machine-Learning/blob/main/Machine_Learning_Lab_Module_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Lab Module 01 - Starting Machine Learning with Structured Data

**Based on:** Dutta, A. *et al.* (2022), *Early Prediction of Diabetes Using an Ensemble of Machine Learning Models*, International Journal of Environmental Research and Public Health, 19, 12378.

This notebook re-implements, step by step, the full pipeline described in the paper above:

1. Loading a real-world, tabular health-survey dataset (structured data, not images/text).
2. Cleaning missing values with three different imputation strategies.
3. Selecting the most useful features with four different feature-selection strategies.
4. Tuning six classical ML classifiers with Grid Search.
5. Building a **weighted ensemble** classifier out of the six base models.
6. Statistically comparing all models (box plots + ANOVA test).

Every code cell is preceded by a short explanation. Cells are intentionally kept small — one idea per cell — so you can run, inspect, and modify them one at a time.


## Dataset Description

The paper introduces a **Diabetes Disease Classification (DDC)** dataset built from the **Bangladesh Demographic and Health Survey (BDHS)**, a large, nationally representative household survey. Two versions are available, collected in 2011 and 2017–2018 respectively. Each row is one survey respondent, and the task is **binary classification**: predict whether the respondent has diabetes (fasting blood glucose ≥ 7.0 mmol/L) or not, using only non-invasive demographic and lifestyle information (no blood test result is used as an input feature).

The 13 predictor features (called `F1`–`F13` in the paper) are a mix of categorical and continuous variables:

| Code | Feature | Type |
|---|---|---|
| F1 | Division (region of residence) | Categorical |
| F2 | Urban / Rural residence | Categorical |
| F3 | Wealth index | Categorical |
| F4 | Sex of household head | Categorical |
| F5 | Age | Continuous |
| F6 | Educational status | Categorical |
| F7 | Occupation type | Categorical |
| F8 | Ate anything recently | Categorical |
| F9 | Had a caffeinated drink | Categorical |
| F10 | Smoked | Categorical |
| F11 | Average systolic blood pressure | Continuous |
| F12 | Average diastolic blood pressure | Continuous |
| F13 | Body Mass Index (BMI) | Continuous |

The target column indicates diabetes status (`1` = diabetic, `0` = non-diabetic).

The original dataset is public and can be downloaded from the authors' repository: `https://github.com/kamruleee51/Diabetes-classification-dataset` (folder `Datasets`, files `BDHS_DDC_2011.xlsx` and `BDHS_DDC_2017_2018.xlsx`). Download **one** of these two files to your computer before running the upload cell below (you are also free to use any other structured/tabular dataset with a binary target — you will just need to adjust the column names in Section 3).


## 1. Environment Setup

This installs the two gradient-boosting libraries used in the paper (`xgboost`, `lightgbm`) plus `openpyxl`, which pandas needs to read `.xlsx` files. Everything else (scikit-learn, pandas, numpy, matplotlib, scipy) is already available in Colab.

In [ ]:
!pip install -q xgboost lightgbm openpyxl

This imports every library we will use throughout the notebook. Grouping imports in one place makes the rest of the notebook easier to read.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.base import clone
from sklearn.impute import KNNImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import roc_auc_score, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from scipy import stats

np.random.seed(42)

## 2. Upload the Dataset

Run this cell and pick the `.xlsx` file you downloaded (`BDHS_DDC_2011.xlsx` or `BDHS_DDC_2017_2018.xlsx`), or your own tabular dataset. This uses Colab's file-upload widget, so it will only work inside Google Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()
FILENAME = list(uploaded.keys())[0]
print("Uploaded file:", FILENAME)

Saving BDHS_DDC_2017_2018.xlsx to BDHS_DDC_2017_2018.xlsx
Uploaded file: BDHS_DDC_2017_2018.xlsx


This reads the uploaded spreadsheet into a pandas DataFrame — the standard table-like structure we will use for the rest of the notebook — and shows its size and first rows.

In [ ]:
df_raw = pd.read_excel(FILENAME)
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (12299, 14)


,HV024,HV025,HV270,HV104,HV105,HV106,SB308,SB311A,SB311B,SB311C,SB333AA,SB333AB,Deabetic,SBBM
0,1,2,1,1,25,1,31,1.0,1.0,1.0,111,58,1,2055
1,1,2,1,2,34,0,0,1.0,1.0,1.0,101,67,1,2481
2,1,2,1,2,35,1,0,1.0,1.0,1.0,100,66,1,1744
3,1,2,1,1,55,2,52,1.0,0.0,1.0,105,71,1,2108
4,1,2,1,2,45,1,15,1.0,1.0,0.0,144,94,1,2455


## 3. Understanding the Data

The original BDHS files use survey codebook names (e.g. `HV024`, `SB333AA`) that differ slightly between the 2011 and 2017–2018 releases. Set `YEAR` below to match the file you uploaded so that the columns get renamed to the friendly `F1`–`F13` + `Target` names used in the paper.

In [ ]:
YEAR = 2017  # change to 2017 if you uploaded BDHS_DDC_2017_2018.xlsx

if YEAR == 2011:
    COLUMN_MAP = {
        "HV024": "F1", "HV025": "F2", "HV270": "F3", "HV104": "F4",
        "HV105": "F5", "HV106": "F6", "SB308": "F7", "SB311A": "F8",
        "SB311B": "F9", "SB311C": "F10", "SB333AA": "F11",
        "SB333AB": "F12", "SHBM": "F13", "Diabetes": "Target",
    }
else:
    COLUMN_MAP = {
        "HV024": "F1", "HV025": "F2", "HV270": "F3", "HV104": "F4",
        "HV105": "F5", "HV106": "F6", "SB308": "F7", "SB311A": "F8",
        "SB311B": "F9", "SB311C": "F10", "SB333AA": "F11",
        "SB333AB": "F12", "SBBM": "F13", "Deabetic": "Target",
    }

df = df_raw.rename(columns=COLUMN_MAP)[list(COLUMN_MAP.values())].copy()
df.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,Target
0,1,2,1,1,25,1,31,1.0,1.0,1.0,111,58,2055,1
1,1,2,1,2,34,0,0,1.0,1.0,1.0,101,67,2481,1
2,1,2,1,2,35,1,0,1.0,1.0,1.0,100,66,1744,1
3,1,2,1,1,55,2,52,1.0,0.0,1.0,105,71,2108,1
4,1,2,1,2,45,1,15,1.0,1.0,0.0,144,94,2455,1


These are the two groups of features we will treat differently later on: `CATEGORICAL_FEATURES` (discrete codes) and `CONTINUOUS_FEATURES` (numeric measurements), matching Table 3 of the paper.

In [ ]:
CATEGORICAL_FEATURES = ["F1", "F2", "F3", "F4", "F6", "F7", "F8", "F9", "F10"]
CONTINUOUS_FEATURES = ["F5", "F11", "F12", "F13"]
ALL_FEATURES = CATEGORICAL_FEATURES + CONTINUOUS_FEATURES
TARGET = "Target"

A quick statistical summary tells us the scale and spread of every column, and `.info()` confirms the data types pandas assigned.

In [ ]:
df.describe()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,Target
count,12299.000000,12299.000000,12299.000000,12299.000000,12299.000000,12299.000000,12299.000000,12292.000000,12292.000000,12292.000000,12299.000000,12299.000000,12299.000000,12299.000000
mean,4.519067,1.642816,3.060086,1.569233,39.847305,1.369786,15.645581,0.325740,0.074113,0.154979,123.346939,81.117001,2328.192617,0.510204
std,2.240941,0.479189,1.430126,0.495204,16.555463,1.035168,17.245654,0.481007,0.293598,0.392941,31.313292,27.006653,924.122039,0.649848
min,1.000000,1.000000,1.000000,1.000000,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,63.000000,38.000000,1219.000000,0.000000
25%,3.000000,1.000000,2.000000,1.000000,26.000000,1.000000,0.000000,0.000000,0.000000,0.000000,109.000000,73.000000,1941.000000,0.000000
50%,4.000000,2.000000,3.000000,2.000000,36.000000,1.000000,13.000000,0.000000,0.000000,0.000000,119.000000,80.000000,2199.000000,0.000000
75%,6.000000,2.000000,4.000000,2.000000,50.000000,2.000000,23.000000,1.000000,0.000000,0.000000,131.000000,87.000000,2503.000000,1.000000
max,8.000000,2.000000,5.000000,2.000000,95.000000,8.000000,99.000000,9.000000,9.000000,9.000000,996.000000,996.000000,9999.000000,2.000000


## 4. Cleaning Missing-Value Codes

Health surveys often mark a missing answer with a special numeric code (e.g. `9999`) instead of leaving the cell empty, so pandas does not automatically recognise them as missing. This cell replaces the known sentinel codes with proper `NaN` values so our imputation methods can find them.

In [ ]:
SENTINEL_CODES = {
    "F5": [], "F11": [996, 998, 999], "F12": [996, 998, 999],
    "F13": [9999], "F7": [99], "F8": [9], "F9": [9], "F10": [9],
}

df_clean = df.copy()
for col, codes in SENTINEL_CODES.items():
    df_clean[col] = df_clean[col].replace(codes, np.nan)

This shows the percentage of missing values per column after cleaning — compare it with the paper's reported 11.25% overall missingness across six of the thirteen features.

In [ ]:
missing_pct = df_clean[ALL_FEATURES].isna().mean().mul(100).round(2)
missing_pct[missing_pct > 0].sort_values(ascending=False)

,0
F13,1.15
F7,0.27
F10,0.09
F9,0.08
F8,0.07
F11,0.01
F12,0.01


## 5. Missing Value Imputation (MVI)

The paper compares three strategies for handling missing values:

- **Case Deletion** – simply drop every row that has at least one missing value.
- **MEDimpute** – replace a missing value with the column's median.
- **KNNimpute** – replace a missing value with a value estimated from the *k* most similar rows.

We implement all three, then measure (via cross-validated AUC) which one works best, exactly like the paper's ablation study in Table 4.


Case deletion: rows with any missing feature are removed entirely.

In [ ]:
def case_deletion(data):
    return data.dropna(subset=ALL_FEATURES).reset_index(drop=True)

Median imputation: each missing cell is filled with the median of its own column.

In [ ]:
def median_impute(data):
    out = data.copy()
    out[ALL_FEATURES] = out[ALL_FEATURES].fillna(out[ALL_FEATURES].median())
    return out

KNN imputation: each missing cell is estimated from the 5 nearest rows (in feature space), using scikit-learn's `KNNImputer`.

In [ ]:
def knn_impute(data, k=5):
    out = data.copy()
    imputer = KNNImputer(n_neighbors=k)
    out[ALL_FEATURES] = imputer.fit_transform(out[ALL_FEATURES])
    return out

This builds the three cleaned versions of the dataset, one per imputation strategy, so we can compare them fairly.

In [ ]:
datasets_mvi = {
    "Case Deletion": case_deletion(df_clean),
    "MEDimpute": median_impute(df_clean),
    "KNNimpute": knn_impute(df_clean),
}
{name: d.shape for name, d in datasets_mvi.items()}

{'Case Deletion': (12118, 14),
 'MEDimpute': (12299, 14),
 'KNNimpute': (12299, 14)}

### 5.1 Comparing the three MVI techniques

This helper trains one classifier with 5-fold cross-validation and returns the mean AUC — our single yardstick for comparing preprocessing choices, just like the paper does.

In [ ]:
def cv_auc(estimator, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for train_idx, test_idx in skf.split(X, y):
        model = clone(estimator)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[test_idx], proba))
    return np.mean(scores), np.std(scores)

These are the six default classifiers used throughout the notebook (matching the paper's GNB, BNB, RF, DT, XGB, LGB), with untuned default hyperparameters for now — tuning comes in Section 7.

In [ ]:
BASE_CLASSIFIERS = {
    "GNB": GaussianNB(),
    "BNB": BernoulliNB(),
    "RF": RandomForestClassifier(random_state=42),
    "DT": DecisionTreeClassifier(random_state=42),
    "XGB": XGBClassifier(eval_metric="logloss", random_state=42),
    "LGB": LGBMClassifier(verbose=-1, random_state=42),
}

This runs every classifier on every MVI-cleaned dataset and collects the mean AUC into a table — the same comparison as Table 4 in the paper.

In [ ]:
mvi_results = []
for mvi_name, data in datasets_mvi.items():
    # Filter out rows where the target is not 0 or 1 to ensure binary classification
    data_filtered = data[data[TARGET].isin([0, 1])]
    X, y = data_filtered[ALL_FEATURES], data_filtered[TARGET]
    for clf_name, clf in BASE_CLASSIFIERS.items():
        mean_auc, std_auc = cv_auc(clf, X, y)
        mvi_results.append({"MVI": mvi_name, "Classifier": clf_name,
                             "AUC_mean": round(mean_auc, 3), "AUC_std": round(std_auc, 3)})

mvi_results_df = pd.DataFrame(mvi_results)
mvi_results_df.pivot(index="Classifier", columns="MVI", values="AUC_mean")

MVI,Case Deletion,KNNimpute,MEDimpute
Classifier,,,
BNB,0.523,0.523,0.523
DT,0.530,0.534,0.534
GNB,0.581,0.565,0.565
LGB,0.603,0.605,0.607
RF,0.599,0.601,0.603
XGB,0.576,0.582,0.586


Based on this comparison (and following the paper's finding), we move forward with **median imputation**. Feel free to change `BEST_MVI` and re-run the rest of the notebook to test another strategy.

In [ ]:
BEST_MVI = "MEDimpute"
data_imputed = datasets_mvi[BEST_MVI]
X_full, y_full = data_imputed[ALL_FEATURES], data_imputed[TARGET]

## 6. Feature Selection (FS)

Not every feature contributes equally to the prediction. The paper ranks features with four different importance-scoring methods and then measures how AUC changes as we keep only the top-*k* ranked features. We implement the same four methods.


RF-based ranking: fit a Random Forest and read off its built-in `feature_importances_` (based on impurity reduction).

In [ ]:
def rf_feature_ranking(X, y):
    model = RandomForestClassifier(random_state=42).fit(X, y)
    return pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

Information-Gain-based ranking: use scikit-learn's `mutual_info_classif`, which measures how much knowing a feature reduces uncertainty about the target — the standard implementation of Information Gain.

In [ ]:
def ig_feature_ranking(X, y):
    scores = mutual_info_classif(X, y, random_state=42)
    return pd.Series(scores, index=X.columns).sort_values(ascending=False)

XGBoost-based ranking: fit an XGBoost model and read off its gain-based `feature_importances_`.

In [ ]:
def xgb_feature_ranking(X, y):
    model = XGBClassifier(eval_metric="logloss", random_state=42).fit(X, y)
    return pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

LightGBM-based ranking: same idea, using LightGBM's own importance scores.

In [ ]:
def lgb_feature_ranking(X, y):
    model = LGBMClassifier(verbose=-1, random_state=42).fit(X, y)
    return pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

This computes all four rankings on the imputed dataset and lays them side by side — the notebook's version of Table 5.

In [ ]:
fs_rankings = {
    "RF": rf_feature_ranking(X_full, y_full),
    "IG": ig_feature_ranking(X_full, y_full),
    "XGB": xgb_feature_ranking(X_full, y_full),
    "LGB": lgb_feature_ranking(X_full, y_full),
}
pd.DataFrame({name: rank / rank.sum() for name, rank in fs_rankings.items()}).round(3)

### 6.1 AUC vs. number of selected features

This helper trains a classifier using only the top-*k* ranked features and returns the cross-validated AUC, letting us trace an AUC-vs-*k* curve for any (ranking, classifier) pair.

In [ ]:
def auc_with_top_k(ranking, k, clf, X, y):
    top_features = ranking.index[:k].tolist()
    mean_auc, _ = cv_auc(clf, X[top_features], y)
    return mean_auc

We now sweep *k* from 2 to 13 features for every FS method, using RF and XGB as example classifiers (the paper does this for all six — we keep two here to stay fast; try adding more from `BASE_CLASSIFIERS`).

In [ ]:
feature_counts = range(2, 14)
example_classifiers = {"RF": BASE_CLASSIFIERS["RF"], "XGB": BASE_CLASSIFIERS["XGB"]}

fs_ablation = {}
for fs_name, ranking in fs_rankings.items():
    for clf_name, clf in example_classifiers.items():
        aucs = [auc_with_top_k(ranking, k, clf, X_full, y_full) for k in feature_counts]
        fs_ablation[f"{fs_name} + {clf_name}"] = aucs

This plots every curve — the same style of chart as Figure 2 in the paper — so you can visually spot which (FS method, feature count) combination peaks highest.

In [ ]:
plt.figure(figsize=(8, 5))
for label, aucs in fs_ablation.items():
    plt.plot(list(feature_counts), aucs, marker="o", label=label)
plt.xlabel("Number of top-ranked features")
plt.ylabel("AUC")
plt.title("AUC vs. number of selected features")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

Following the paper's conclusion, we keep the **top-5 RF-ranked features** for the rest of the notebook. Change `N_FEATURES` or `FS_METHOD` to experiment with a different choice.

In [ ]:
FS_METHOD = "RF"
N_FEATURES = 5
SELECTED_FEATURES = fs_rankings[FS_METHOD].index[:N_FEATURES].tolist()
print("Selected features:", SELECTED_FEATURES)

X, y = X_full[SELECTED_FEATURES], y_full

## 7. Hyperparameter Optimization with Grid Search

Each classifier has "knobs" (hyperparameters) that control how it learns. The paper tunes these with **Grid Search**: try every combination from a small predefined set of values, and keep the one with the best cross-validated AUC. The grids below are deliberately small so this runs quickly in class; feel free to widen them.


This defines a small hyperparameter grid for every classifier.

In [ ]:
PARAM_GRIDS = {
    "GNB": {"var_smoothing": [1e-9, 1e-2]},
    "BNB": {"alpha": [0.5, 1.0]},
    "RF": {"n_estimators": [100, 200], "max_depth": [3, 5, None]},
    "DT": {"criterion": ["gini", "entropy"], "max_depth": [3, 5, None]},
    "XGB": {"n_estimators": [100, 200], "max_depth": [3, 5]},
    "LGB": {"n_estimators": [50, 100], "num_leaves": [15, 25]},
}

This runs `GridSearchCV` (3-fold, scored by AUC) for every classifier and stores the best-performing estimator for each — the notebook's version of Table 6.

In [ ]:
best_estimators = {}
tuning_results = []

for clf_name, clf in BASE_CLASSIFIERS.items():
    grid = GridSearchCV(clf, PARAM_GRIDS[clf_name], scoring="roc_auc", cv=3, n_jobs=-1)
    grid.fit(X, y)
    best_estimators[clf_name] = grid.best_estimator_
    tuning_results.append({"Classifier": clf_name, "Best Params": grid.best_params_,
                            "Best AUC (3-fold)": round(grid.best_score_, 3)})

pd.DataFrame(tuning_results)

## 8. Evaluating Individual Classifiers

We now score every tuned classifier with the same 5-fold split, using the four metrics used in the paper: Sensitivity (Sn), Specificity (Sp), Accuracy (Acc), and AUC. Using the **same** folds for every classifier (and later for the ensembles) is essential for a fair comparison and for the statistical test in Section 10.


This creates one fixed set of 5 train/test splits (`FOLD_INDICES`) that every model below will reuse.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
FOLD_INDICES = list(skf.split(X, y))

This turns raw predicted probabilities into Sn, Sp, Acc and AUC for one fold, following Equations (4)-(6) of the paper.

In [ ]:
def metrics_from_proba(proba, y_true, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sn = tp / (tp + fn) if (tp + fn) else 0.0
    sp = tn / (tn + fp) if (tn + fp) else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn)
    auc = roc_auc_score(y_true, proba)
    return sn, sp, acc, auc

This trains one classifier across all 5 fixed folds and returns, per fold: the predicted probabilities, the true labels, and the four metrics — everything we need both for the results table and for the ensemble later.

In [ ]:
def evaluate_classifier(estimator, X, y, fold_indices):
    proba_folds, y_folds = [], []
    sn_list, sp_list, acc_list, auc_list = [], [], [], []
    for train_idx, test_idx in fold_indices:
        model = clone(estimator)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        y_test = y.iloc[test_idx].values

        sn, sp, acc, auc = metrics_from_proba(proba, y_test)
        proba_folds.append(proba); y_folds.append(y_test)
        sn_list.append(sn); sp_list.append(sp); acc_list.append(acc); auc_list.append(auc)

    return {"proba_folds": proba_folds, "y_folds": y_folds,
            "Sn": sn_list, "Sp": sp_list, "Acc": acc_list, "AUC": auc_list}

This runs `evaluate_classifier` for all six tuned models and keeps everything in `individual_results`, which we will reuse for the ensemble and for the statistical analysis.

In [ ]:
individual_results = {
    name: evaluate_classifier(model, X, y, FOLD_INDICES)
    for name, model in best_estimators.items()
}

This summarises each classifier's five fold-scores into mean ± std, reproducing the top half of Table 7 in the paper.

In [ ]:
def summarize(results_dict):
    rows = []
    for name, res in results_dict.items():
        rows.append({
            "Model": name,
            "Sn": f"{np.mean(res['Sn']):.3f} ± {np.std(res['Sn']):.3f}",
            "Sp": f"{np.mean(res['Sp']):.3f} ± {np.std(res['Sp']):.3f}",
            "Acc": f"{np.mean(res['Acc']):.3f} ± {np.std(res['Acc']):.3f}",
            "AUC": f"{np.mean(res['AUC']):.3f} ± {np.std(res['AUC']):.3f}",
        })
    return pd.DataFrame(rows)

summarize(individual_results)

## 9. Weighted Ensemble Classifier

The paper's key idea: combine several classifiers by **weighting each one's predicted probability by its own AUC** (Equation 3), then normalising. Classifiers that are individually more reliable get more say in the final decision.


Each classifier's weight is simply its mean AUC from Section 8.

In [ ]:
weights = {name: np.mean(res["AUC"]) for name, res in individual_results.items()}
weights

This combines any chosen subset of classifiers fold-by-fold, using the *same* stored predictions from Section 8 (no retraining needed), and returns the ensemble's per-fold metrics — implementing Equation (3) of the paper.

In [ ]:
def evaluate_ensemble(member_names, individual_results, weights):
    n_folds = len(individual_results[member_names[0]]["proba_folds"])
    sn_list, sp_list, acc_list, auc_list = [], [], [], []

    for f in range(n_folds):
        weighted_pos = sum(weights[m] * individual_results[m]["proba_folds"][f] for m in member_names)
        weighted_neg = sum(weights[m] * (1 - individual_results[m]["proba_folds"][f]) for m in member_names)
        proba_ensemble = weighted_pos / (weighted_pos + weighted_neg)
        y_test = individual_results[member_names[0]]["y_folds"][f]

        sn, sp, acc, auc = metrics_from_proba(proba_ensemble, y_test)
        sn_list.append(sn); sp_list.append(sp); acc_list.append(acc); auc_list.append(auc)

    return {"Sn": sn_list, "Sp": sp_list, "Acc": acc_list, "AUC": auc_list}

This tries the same ensemble combinations explored in the paper's ablation study: pairs, groups of four, and all six classifiers together.

In [ ]:
ensemble_combos = {
    "GNB + BNB": ["GNB", "BNB"],
    "RF + DT": ["RF", "DT"],
    "XGB + LGB": ["XGB", "LGB"],
    "GNB + BNB + DT + RF": ["GNB", "BNB", "DT", "RF"],
    "GNB + BNB + XGB + LGB": ["GNB", "BNB", "XGB", "LGB"],
    "DT + RF + XGB + LGB": ["DT", "RF", "XGB", "LGB"],
    "All Six": ["GNB", "BNB", "RF", "DT", "XGB", "LGB"],
}

ensemble_results = {
    combo_name: evaluate_ensemble(members, individual_results, weights)
    for combo_name, members in ensemble_combos.items()
}

This lays the individual models and every ensemble combination side by side — the notebook's full reproduction of Table 7, so you can identify the best-performing model overall.

In [ ]:
all_results = {**individual_results, **ensemble_results}
summarize(all_results)

## 10. Result Analysis: Box Plots & ANOVA Test

Finally, we visualise the spread of AUC across the 5 folds for every model, then run a statistical **ANOVA test** to check whether the differences between models are significant — exactly like Figure 3 and the ANOVA test described in the paper.


This draws a box-and-whisker plot of the 5 fold-level AUC scores for every model (individual classifiers and ensembles together).

In [ ]:
plt.figure(figsize=(10, 5))
labels = list(all_results.keys())
auc_data = [all_results[name]["AUC"] for name in labels]
plt.boxplot(auc_data, tick_labels=labels, showmeans=True)
plt.xticks(rotation=75)
plt.ylabel("AUC (5-fold CV)")
plt.title("Fold-wise AUC distribution per model")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

This runs a one-way ANOVA across all models' fold-level AUC scores: a small p-value (≤ 0.05) means the models' average AUCs are *not* all equal, i.e. at least one model is genuinely different.

In [ ]:
anova_stat, anova_p = stats.f_oneway(*auc_data)
print(f"ANOVA F-statistic = {anova_stat:.3f}, p-value = {anova_p:.3e}")
print("Significant difference between models" if anova_p <= 0.05 else "No significant difference between models")

If the ANOVA is significant, this post-hoc test compares the best-looking model against every other model individually (independent t-test) to confirm it is a genuine improvement, not folding noise.

In [ ]:
best_model = max(all_results, key=lambda name: np.mean(all_results[name]["AUC"]))
print("Best model by mean AUC:", best_model)

for name in all_results:
    if name == best_model:
        continue
    t_stat, p_val = stats.ttest_ind(all_results[best_model]["AUC"], all_results[name]["AUC"])
    print(f"{best_model} vs {name}: p-value = {p_val:.3f}")

## 11. Conclusion & Exercises

You have now reproduced the full pipeline of the paper: **missing-value imputation → feature selection → hyperparameter tuning → individual classifiers → weighted ensemble → statistical validation**.

**Try it yourself:**
1. Re-run Section 5 with `BEST_MVI = "KNNimpute"` — does the ranking of classifiers change?
2. Re-run Section 6 with `FS_METHOD = "XGB"` and `N_FEATURES = 8` — does AUC improve?
3. Widen the grids in `PARAM_GRIDS` (Section 7) — does tuning longer help?
4. Add more classifiers to `ensemble_combos` (Section 9) — can you beat `DT + RF + XGB + LGB`, the paper's best ensemble?
5. Upload the *other* year's file (2011 vs 2017–2018) and compare — does the best model stay the same across datasets, as discussed in the paper's Section 3.4.3?
